# 07 — Traffic consistency audit

**CPU runtime → Runtime → Run all → upload your completed notebook 06 results ZIP when prompted.**

This notebook checks whether saved adversarial aggregate records pass one additional necessary condition: **a positive sum requires a positive maximum in the same aggregate**. It audits all 180 pairs on clean training/development records and all exported candidate pools, then looks for alternative evasions when a previously selected vector fails.

It creates a new results ZIP. Keep the original ZIP and all previous notebooks. No new attacks or model training are needed.


## 1. Create a new session

Leave `INPUT_ZIP` empty to use the upload button. If the ZIP is already on this runtime, enter its absolute path. Use the same notebook 06 ZIP that you previously shared.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

INPUT_ZIP = "" #@param {type:"string"}
INPUT_ZIP = os.environ.get("PHD_INPUT_ZIP", INPUT_ZIP)
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
if os.environ.get("PHD_NO_DOWNLOAD") == "1":
    DOWNLOAD_RESULTS = False
REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis_TrafficConsistency")).resolve()
HOST_PYTHON, HOST_VERSION = sys.executable, sys.version
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} is not a git checkout. Choose a new workspace.")
for required in ["scripts/bootstrap.py", "baseline/run_traffic_consistency_audit.py", "notebooks/07_Traffic_Consistency_Audit.ipynb"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the updated repository.")

# Existing checkouts are preserved; no pull/reset/checkout is performed.
def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("traffic_consistency")
SESSION.mkdir(parents=True, exist_ok=False)
source_paths = [p for folder in ["baseline", "scripts", "tests"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/07_Traffic_Consistency_Audit.ipynb", ROOT / "docs/TRAFFIC_CONSISTENCY_GUIDE.md"]
context = {
    "project_repository": REPO_URL,
    "project_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
    "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    "experiment": "saved_pool_traffic_consistency_audit", "evaluation_device": "cpu",
}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
for p in source_paths:
    target = SESSION / "source_snapshot" / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Export partial logs with the last cell.")

print("Project commit:", context["project_commit"])
print("New session:", SESSION)


In [ ]:
if INPUT_ZIP.strip():
    INPUT = Path(INPUT_ZIP).expanduser().resolve()
else:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("Set INPUT_ZIP to your completed notebook 06 results ZIP.")
    upload_dir = SESSION / "upload"
    upload_dir.mkdir(exist_ok=False)
    previous_cwd = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
    finally:
        os.chdir(previous_cwd)
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one completed notebook 06 results ZIP.")
    INPUT = upload_dir / next(iter(uploaded))
    del uploaded
if not INPUT.is_file() or INPUT.suffix.lower() != ".zip":
    raise ValueError("INPUT_ZIP must identify an existing ZIP file.")
print("Input evidence:", INPUT.name)
print("Input SHA256:", hashlib.sha256(INPUT.read_bytes()).hexdigest())


## 2. Prepare the environment and pinned assets

The notebook installs isolated Python 3.8.20, NumPy 1.23.5 and CPU PyTorch without changing the notebook kernel. A GPU is unnecessary for this audit. A fresh runtime downloads approximately 348 MB of aggregate feature data plus the small checkpoints.

The runner reads only the native training/development prefix for clean controls. It restores the earlier frozen compatibility scaler; it does not fit a new scaler. That earlier scaler used all feature rows, an inherited limitation that prevents treating this as a leakage-free final test.


In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
setup_copy = SESSION / unique_name("setup_report")
shutil.copy2(ROOT / ".cache/setup_report.json", setup_copy.with_suffix(".json"))
print("Pinned audit interpreter:", PYTHON)


In [ ]:
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
manifest_copy = SESSION / (unique_name("download_manifest") + ".json")
shutil.copy2(ROOT / "assets/download_manifest.json", manifest_copy)


## 3. Audit the saved evidence

Checks run in this order:
1. Verify the ZIP, model/data hashes, feature order, split, saved scaler and baseline/in-loop pairing.
2. Check all 143,046 training/development records, separately by partition and benign/malicious label, in parsed float64 and model float32.
3. Recompute native validity, distances and model predictions for every exported stage pool and each method's effective final result.
4. Compare the original reported result, a fresh native-valid pool selection, and selection with the added sum/maximum rule.

A failed chosen candidate can be replaced by a valid saved alternative. Each final matrix is rechecked with the unchanged native checker. In-loop methods remain standalone; their more expensive union with a separate original attack is excluded.

This scans exported stage populations, not unexported intermediate optimizer states. The rule is a post-hoc development diagnostic; it does not establish complete traffic realizability.


In [ ]:
RUN = SESSION / "audit"
run_command([PYTHON, ROOT / "baseline/run_traffic_consistency_audit.py",
             "--input-zip", INPUT, "--out", RUN,
             "--repo", ROOT / "vendor/tabularbench", "--assets", ROOT / "assets"], "audit.log")
print("Completed traffic consistency audit:", RUN)


## 4. Read the comparison

`published_newly_evaded` reproduces notebook 06. `native_final_newly_evaded` selects from all saved sources under the original rules. `native_plus_support_final_newly_evaded` adds the sum/maximum check. These last two columns use the same new selection procedure.

`flagged_published_evasions_with_passing_alternative` shows why checking only the earlier final vector can undercount surviving evasions. Evasion rates use initially detected malicious records as their denominator; they are not overall IDS accuracy.


In [ ]:
import pandas as pd
from IPython.display import display

protocol = json.loads((RUN / "protocol.json").read_text())
results = json.loads((RUN / "results.json").read_text())
assert protocol["status"] == results["status"] == "completed"
assert not protocol["new_attack_generation"] and not protocol["test_partition_evaluated"]
print("Clean training/development controls:")
display(pd.read_csv(RUN / "clean_controls.csv"))
comparison = pd.read_csv(RUN / "comparison.csv")
display(comparison[["model", "seed", "method", "clean_detected", "published_newly_evaded",
                    "published_evasions_flagged", "native_final_newly_evaded",
                    "native_plus_support_final_newly_evaded",
                    "flagged_published_evasions_with_passing_alternative"]])
print("Audit counts:", results["counts"])
print("Audit runtime (minutes):", round(results["wall_seconds"] / 60, 2))
if not results["clean_controls"]["rule_passed_all_clean_controls"]:
    print("Clean records failed the added rule. Investigate the recorded cases before interpreting traffic consistency.")
print("Passing this one condition does not prove realizable or still-malicious network traffic.")


## 5. Download the new evidence ZIP

The ZIP contains the input evidence, rule protocol, clean-control counts, rescored source masks, per-record selections, final vectors, source snapshots and logs. The dataset, model weights and Python environment are excluded.

**Download and share this new ZIP.** If an earlier cell fails, run this export cell separately to preserve partial outputs and logs.


In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
import zipfile
archive = export_base.with_suffix(".zip")
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(SESSION.rglob("*")):
        if path.is_file() and "upload" not in path.relative_to(SESSION).parts:
            z.write(path, path.relative_to(SESSION.parent))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


## What this contributes now

This audit tests whether an apparent feature-space attack improvement survives a documented necessary consistency condition. It is groundwork for defining a defensible threat model and evaluation protocol.

It does not train a stronger IDS, establish a novel defense, or replay traffic. Keep negative findings. The next research decision depends on whether useful evasions survive and whether their traffic semantics can be justified.
